<a href="https://colab.research.google.com/github/harinijk/YoutubeTrends/blob/main/YoutubeTrends.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Section 1. Data Set & Problem Definition

In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from torch.utils.data import Dataset, DataLoader
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.auto import tqdm
from sklearn.metrics import classification_report

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print(f"Using device: {device}")

In [ ]:
df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "meruvakodandasuraj/youtube-trending-videos-20202026",
    "trending_videos.csv")

print("shape", df.shape)
print("Columns", df.columns)
display(df.head())

In [ ]:
df = df.drop_duplicates()
df = df.dropna(subset=["title","category","engagement_score","clickbait_score","has_caps_title","tag_count"])
df = df[df["engagement_score"] >= 0]


In [ ]:
threshold = df["engagement_score"].median()

df["high_engagement"] = (df["engagement_score"] > threshold).astype(int)
print("Engagement threshold", threshold)
print(df["high_engagement"].value_counts())
print(df["high_engagement"].value_counts(normalize=True))

corr = df.corr(numeric_only=True)["engagement_score"].sort_values(ascending=False)
print("Correlation with engagement_score")
print(corr)


In [ ]:
low_cutoff = df["engagement_score"].quantile(0.30)
high_cutoff = df["engagement_score"].quantile(0.70)

df = df[(df["engagement_score"] <= low_cutoff) | (df["engagement_score"] >= high_cutoff)].copy()

df["high_engagement"] = (df["engagement_score"] >= high_cutoff).astype(int)


In [ ]:
plt.figure(figsize=(7, 4))
sns.histplot(df["engagement_score"], bins=50, kde=True)
plt.axvline(low_cutoff, linestyle="--", label="Low cutoff")
plt.axvline(high_cutoff, linestyle="--", label="High cutoff")
plt.title("Engagement Score Distribution")
plt.xlabel("Engagement Score")
plt.ylabel("Number of Videos")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(5, 4))
sns.countplot(data=df, x="high_engagement")
plt.title("Low vs High Engagement Class Balance")
plt.xlabel("High Engagement")
plt.ylabel("Count")
plt.xticks([0, 1], ["Low", "High"])
plt.show()


In [ ]:
features_to_check = ["clickbait_score","has_caps_title","tag_count","title_length","subscriber_count"]

for col in features_to_check:
    if col in df.columns:
        plt.figure(figsize=(5, 3))
        sns.boxplot(data=df, x="high_engagement", y=col)
        plt.title(f"{col} by Engagement Class")
        plt.xlabel("High Engagement")
        plt.ylabel(col)
        plt.xticks([0, 1], ["Low", "High"])
        plt.show()

In [ ]:
df = df[["title","category","clickbait_score","has_caps_title","tag_count","engagement_score","high_engagement"]]

display(df.head())

In [ ]:
train_df, temp_df = train_test_split(df,test_size=0.30,random_state=42,stratify=df["high_engagement"])
val_df, test_df = train_test_split(temp_df,test_size=0.50,random_state=42,stratify=temp_df["high_engagement"])

train_df["text_input"] = ("Title: " + train_df["title"].astype(str) +" Category: " + train_df["category"].astype(str) +" ClickbaitScore: " + train_df["clickbait_score"].astype(str) +" TagsCount: " + train_df["tag_count"].astype(str))
val_df["text_input"] = ("Title: " + val_df["title"].astype(str) + " Category: " + val_df["category"].astype(str) + " ClickbaitScore: " + val_df["clickbait_score"].astype(str) + " TagsCount: " + val_df["tag_count"].astype(str))
test_df["text_input"] = ("Title: " + test_df["title"].astype(str) +" Category: " + test_df["category"].astype(str) +" ClickbaitScore: " + test_df["clickbait_score"].astype(str) +" TagsCount: " + test_df["tag_count"].astype(str))

print("Split sizes:")
print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

print("\nTrain class distribution:")
print(train_df["high_engagement"].value_counts(normalize=True))

print("\nValidation class distribution:")
print(val_df["high_engagement"].value_counts(normalize=True))

print("\nTest class distribution:")
print(test_df["high_engagement"].value_counts(normalize=True))

In [ ]:
print("Feature summary:")
display(df.describe(include="all").transpose())

Section 2. Embedding Model + Transfer Learning

In [ ]:
text_col = "text_input"
target_col = "high_engagement"

X_train_text = train_df[text_col].astype(str).values
X_val_text = val_df[text_col].astype(str).values
X_test_text = test_df[text_col].astype(str).values

y_train = train_df[target_col].values.astype(np.float32)
y_val = val_df[target_col].values.astype(np.float32)
y_test = test_df[target_col].values.astype(np.float32)

print("Training text samples:", len(X_train_text))
print("Validation text samples:", len(X_val_text))
print("Testing text samples:", len(X_test_text))

In [ ]:
embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"

sentence_backbone = SentenceTransformer(embedding_model_name)
print("Embedding model:", embedding_model_name)

X_train_embed = sentence_backbone.encode(X_train_text,convert_to_numpy=True,show_progress_bar=True)

X_val_embed = sentence_backbone.encode(X_val_text,convert_to_numpy=True,show_progress_bar=True)

X_test_embed = sentence_backbone.encode(X_test_text,convert_to_numpy=True,show_progress_bar=True)

X_train_full = X_train_embed
X_val_full = X_val_embed
X_test_full = X_test_embed

In [ ]:
class EngagementDataset(Dataset):
    def __init__(self, features, targets):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.targets = torch.tensor(targets, dtype=torch.float32)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx], self.targets[idx]


batch_size = 128

train_dataset = EngagementDataset(X_train_full, y_train)
val_dataset = EngagementDataset(X_val_full, y_val)
test_dataset = EngagementDataset(X_test_full, y_test)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
class EngagementClassifier(nn.Module):
    def __init__(self, input_size):
        super(EngagementClassifier, self).__init__()

        self.classifier = nn.Sequential(
            nn.Linear(input_size, 64),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(64, 16),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(16, 1)
        )

    def forward(self, x):
        return self.classifier(x).squeeze(1)


input_size = X_train_full.shape[1]

engagement_model = EngagementClassifier(input_size).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(engagement_model.parameters(),lr=0.001,weight_decay=1e-4)

In [ ]:
num_epochs = 30

train_losses = []
val_losses = []
val_accuracies = []

best_val_loss = float("inf")
patience = 10
patience_counter = 0
best_model_state = None

for epoch in range(num_epochs):
    engagement_model.train()
    total_train_loss = 0

    for features, targets in train_loader:
        features = features.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()

        logits = engagement_model(features)
        loss = criterion(logits, targets)

        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / len(train_loader)
    train_losses.append(avg_train_loss)

    engagement_model.eval()
    total_val_loss = 0
    val_preds = []
    val_true = []

    with torch.no_grad():
        for features, targets in val_loader:
            features = features.to(device)
            targets = targets.to(device)

            logits = engagement_model(features)
            loss = criterion(logits, targets)

            total_val_loss += loss.item()

            probs = torch.sigmoid(logits)
            preds = (probs > 0.5).float()

            val_preds.extend(preds.cpu().numpy())
            val_true.extend(targets.cpu().numpy())

    avg_val_loss = total_val_loss / len(val_loader)
    val_losses.append(avg_val_loss)

    val_acc = accuracy_score(val_true, val_preds)
    val_accuracies.append(val_acc)
    print(f"Epoch {epoch} | "
    f"Train Loss: {avg_train_loss:.4f} | "
    f"Val Loss: {avg_val_loss:.4f} | "
    f"Val Accuracy: {val_acc:.4f}")

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        best_model_state = engagement_model.state_dict()
        patience_counter = 0
    else:
        patience_counter += 1

    if patience_counter >= patience:
        print("Early stopping")
        break

engagement_model.load_state_dict(best_model_state)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Validation Loss")
plt.title("Training and Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("BCE Loss")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(val_accuracies, label="Validation Accuracy")
plt.title("Validation Accuracy Over Epochs")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
engagement_model.eval()

all_preds = []
all_probs = []
all_targets = []

with torch.no_grad():
    for features, targets in test_loader:
        features = features.to(device)

        logits = engagement_model(features)
        probs = torch.sigmoid(logits)
        preds = (probs > 0.5).float()

        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
        all_targets.extend(targets.numpy())

all_preds = np.array(all_preds)
all_probs = np.array(all_probs)
all_targets = np.array(all_targets)

accuracy = accuracy_score(all_targets, all_preds)
precision = precision_score(all_targets, all_preds)
recall = recall_score(all_targets, all_preds)
f1 = f1_score(all_targets, all_preds)

print("Test Results:")
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

In [ ]:
cm = confusion_matrix(all_targets, all_preds)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Low Engagement", "High Engagement"],
    yticklabels=["Low Engagement", "High Engagement"]
)

plt.title("Confusion Matrix: Sentence Embeddings + MLP")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()

In [ ]:
results_df = test_df.copy()

results_df["actual_label"] = all_targets
results_df["predicted_label"] = all_preds
results_df["predicted_probability_high"] = all_probs

failure_cases = results_df[
    results_df["actual_label"] != results_df["predicted_label"]
]

print("Number of failure cases:", len(failure_cases))

display(failure_cases[[
    "title",
    "category",
    "engagement_score",
    "actual_label",
    "predicted_label",
    "predicted_probability_high"
]].head(10))

In [ ]:
failure_cases["confidence"] = abs(failure_cases["predicted_probability_high"] - 0.5)

confident_wrong = failure_cases.sort_values("confidence",ascending=False)

display(confident_wrong[[
    "title",
    "category",
    "engagement_score",
    "actual_label",
    "predicted_label",
    "predicted_probability_high",
    "confidence"
]].head(10))

Section 3. Large-Language Model and Section 4. Evaluation

In [ ]:
qwen_model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(qwen_model_name)

qwen_model = AutoModelForCausalLM.from_pretrained( qwen_model_name, torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32, device_map="auto")
qwen_model.eval()

In [ ]:
qwen_prompt_template = """
You are a binary classifier.

Predict whether this YouTube video will have high engagement.

Respond with only: 0 or 1

0 = Low Engagement
1 = High Engagement

Title: {title}
Category: {category}
"""
def parse_label(text: str):
    cleaned = text.strip()

    if cleaned == "1":
        return 1
    elif cleaned == "0":
        return 0

    return "unknown"

def query_qwen(prompt):
    messages = [{"role": "system","content": "You are a careful classifier. Return only 0 or 1."},{"role": "user","content": prompt}]

    text = tokenizer.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)

    inputs = tokenizer(text, return_tensors="pt").to(qwen_model.device)

    with torch.no_grad():
        outputs = qwen_model.generate(**inputs,max_new_tokens=1,do_sample=False)

    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:],skip_special_tokens=True).strip()

    return response

sample_size = min(200, len(test_df))

qwen_df = test_df.sample(n=sample_size, random_state=42).copy()
qwen_records = []

print(f"Evaluating Qwen on {sample_size} test examples")

for _, row in tqdm(qwen_df.iterrows(), total=len(qwen_df)):
    prompt = qwen_prompt_template.format(title=row["title"],category=row["category"])

    raw_output = query_qwen(prompt)
    pred_label = parse_label(raw_output)

    qwen_records.append({
        "title": row["title"],
        "category": row["category"],
        "engagement_score": row["engagement_score"],
        "true_label": int(row["high_engagement"]),
        "pred_label": pred_label,
        "raw_output": raw_output
    })

qwen_results = pd.DataFrame(qwen_records)

display(qwen_results.head())

In [ ]:
unknown_mask = qwen_results["pred_label"] == "unknown"

print("Total Qwen predictions:", len(qwen_results))
print("Unknown predictions:", unknown_mask.sum())

qwen_valid = qwen_results[~unknown_mask].copy()

qwen_valid["pred_label"] = qwen_valid["pred_label"].astype(int)
qwen_valid["true_label"] = qwen_valid["true_label"].astype(int)

y_true_qwen = qwen_valid["true_label"].values
y_pred_qwen = qwen_valid["pred_label"].values

qwen_accuracy = accuracy_score(y_true_qwen, y_pred_qwen)
qwen_precision = precision_score(y_true_qwen, y_pred_qwen)
qwen_recall = recall_score(y_true_qwen, y_pred_qwen)
qwen_f1 = f1_score(y_true_qwen, y_pred_qwen)

print("Qwen Test Results:")
print("Accuracy:", qwen_accuracy)
print("Precision:", qwen_precision)
print("Recall:", qwen_recall)
print("F1 Score:", qwen_f1)

In [ ]:
cm_qwen = confusion_matrix(y_true_qwen, y_pred_qwen)

plt.figure(figsize=(6, 5))
sns.heatmap(cm_qwen,annot=True,fmt="d",cmap="Oranges", xticklabels=["Low Engagement", "High Engagement"],yticklabels=["Low Engagement", "High Engagement"])

plt.title("Confusion Matrix: Qwen Prompting Baseline")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()

In [ ]:
comparison_df = pd.DataFrame({
    "Model": ["SentenceTransformer + MLP","Qwen"],
    "Accuracy": [accuracy, qwen_accuracy],
    "Precision": [precision,qwen_precision],
    "Recall": [recall,qwen_recall],
    "F1 Score": [f1,qwen_f1]
})

display(comparison_df)

In [ ]:
comparison_df = pd.DataFrame({
    "Model": ["SentenceTransformer + MLP","Qwen/Qwen2.5-0.5B-Instruct"],
    "Accuracy": [accuracy, qwen_accuracy],
    "Precision": [precision,qwen_precision],
    "Recall": [recall,qwen_recall],
    "F1 Score": [f1,qwen_f1]
})

display(comparison_df)

In [ ]:
plot_df = comparison_df.melt(id_vars="Model",var_name="Metric",value_name="Score")

plt.figure(figsize=(10, 5))

sns.barplot(data=plot_df,x="Metric",y="Score",hue="Model")

plt.title("Embedding Model vs Qwen LLM")
plt.ylim(0, 1)
plt.ylabel("Score")
plt.xlabel("Metric")

plt.show()

In [ ]:
print("SentenceTransformer + MLP Classification Report")

print(classification_report(all_targets,all_preds,target_names=["Low Engagement","High Engagement"],zero_division=0))
print("Qwen Classification Report")
print(classification_report(y_true_qwen,y_pred_qwen,target_names=["Low Engagement","High Engagement"],zero_division=0))

In [ ]:
embedding_failures = results_df[results_df["actual_label"] != results_df["predicted_label"]]

print("Embedding failure cases:", len(embedding_failures))

display(embedding_failures[["title","category","engagement_score","actual_label","predicted_label"]].head(10))

In [ ]:
qwen_failures = qwen_valid[qwen_valid["true_label"] != qwen_valid["pred_label"]]
print("Qwen failure cases:", len(qwen_failures))

display(qwen_failures[["title","category","engagement_score","true_label","pred_label","raw_output"]].head(10))